# Segment 4 — LiDAR Classified Points to Objects

This notebook converts **classified LiDAR points** into structured objects.

Pipeline:

`classified points → separate classes → DBSCAN clustering → object properties → JSON output`

For now, the notebook uses synthetic classified LiDAR data so Segment 4 can be tested independently. Later, replace the sample data cell with the real output from Segment 3.


## 1. Install dependencies

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN

## 3. Define semantic classes

In [ ]:
CLASS_NAMES = {
    0: "road",
    1: "vehicle",
    2: "pedestrian",
    3: "pole",
    4: "wall",
    5: "grass",
    6: "rock"
}

DYNAMIC_PROBABILITY = {
    "vehicle": 0.90,
    "pedestrian": 0.98,
    "pole": 0.01,
    "wall": 0.01,
    "road": 0.00,
    "grass": 0.05,
    "rock": 0.02
}

## 4. Create example classified LiDAR data

Each LiDAR point has:

- `x, y, z`
- semantic class label
- classification confidence

Replace this cell later with the real output of Segment 3.


In [ ]:
np.random.seed(42)

# Vehicle 1
vehicle1 = np.random.normal(
    loc=[10, 5, 1],
    scale=[0.8, 0.4, 0.3],
    size=(300, 3)
)

# Vehicle 2
vehicle2 = np.random.normal(
    loc=[25, -4, 1],
    scale=[0.8, 0.4, 0.3],
    size=(300, 3)
)

# Pedestrian
pedestrian = np.random.normal(
    loc=[18, 2, 1],
    scale=[0.15, 0.15, 0.45],
    size=(100, 3)
)

# Pole
pole = np.random.normal(
    loc=[30, 8, 2],
    scale=[0.07, 0.07, 0.8],
    size=(100, 3)
)

points = np.vstack([
    vehicle1,
    vehicle2,
    pedestrian,
    pole
])

labels = np.concatenate([
    np.full(len(vehicle1), 1),
    np.full(len(vehicle2), 1),
    np.full(len(pedestrian), 2),
    np.full(len(pole), 3)
])

confidence = np.random.uniform(
    0.85,
    0.99,
    len(points)
)

print("Total points:", len(points))
print("Points shape:", points.shape)
print("Labels shape:", labels.shape)
print("Confidence shape:", confidence.shape)

## 5. Visualize classified points

In [ ]:
plt.figure(figsize=(10, 7))

for class_id, class_name in CLASS_NAMES.items():
    mask = labels == class_id

    if np.any(mask):
        plt.scatter(
            points[mask, 0],
            points[mask, 1],
            s=8,
            label=class_name
        )

plt.xlabel("X (meters)")
plt.ylabel("Y (meters)")
plt.title("Segment 3 Classified LiDAR Points")
plt.legend()
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()

## 6. Separate points by semantic class

In [ ]:
def separate_classes(points, labels, confidence):
    class_points = {}

    for class_id, class_name in CLASS_NAMES.items():
        mask = labels == class_id

        if np.any(mask):
            class_points[class_name] = {
                "points": points[mask],
                "confidence": confidence[mask]
            }

    return class_points


class_points = separate_classes(points, labels, confidence)

for class_name, data in class_points.items():
    print(f"{class_name:12s}: {len(data['points'])} points")

## 7. DBSCAN clustering

DBSCAN groups nearby semantic points into individual object instances.

For example, 600 points labelled `vehicle` can become:

- Vehicle 1
- Vehicle 2


In [ ]:
def cluster_points(points, eps=1.0, min_samples=10):
    if len(points) == 0:
        return np.array([])

    clustering = DBSCAN(
        eps=eps,
        min_samples=min_samples
    )

    return clustering.fit_predict(points)

## 8. Extract clusters

In [ ]:
def extract_clusters(
    points,
    confidences,
    eps=1.0,
    min_samples=10
):
    cluster_labels = cluster_points(
        points,
        eps=eps,
        min_samples=min_samples
    )

    clusters = []

    for cluster_id in sorted(set(cluster_labels)):
        # DBSCAN uses -1 for noise
        if cluster_id == -1:
            continue

        mask = cluster_labels == cluster_id

        clusters.append({
            "cluster_id": int(cluster_id),
            "points": points[mask],
            "confidence": confidences[mask]
        })

    return clusters

## 9. Calculate object position, dimensions and confidence

In [ ]:
def calculate_object_properties(cluster):
    pts = cluster["points"]
    conf = cluster["confidence"]

    # Object center
    center = np.mean(pts, axis=0)

    # Axis-aligned bounding box
    min_xyz = np.min(pts, axis=0)
    max_xyz = np.max(pts, axis=0)

    dimensions = max_xyz - min_xyz

    # Mean point confidence
    object_confidence = np.mean(conf)

    return {
        "position": center,
        "dimensions": dimensions,
        "confidence": object_confidence,
        "num_points": len(pts)
    }

## 10. Detect objects

In [ ]:
def detect_objects(
    points,
    labels,
    confidence,
    eps=1.0,
    min_samples=10
):
    objects = []

    class_points = separate_classes(
        points,
        labels,
        confidence
    )

    object_id = 0

    # Terrain is handled separately and not clustered as discrete objects.
    terrain_classes = {"road", "grass", "rock"}

    for class_name, data in class_points.items():

        if class_name in terrain_classes:
            continue

        clusters = extract_clusters(
            data["points"],
            data["confidence"],
            eps=eps,
            min_samples=min_samples
        )

        for cluster in clusters:
            properties = calculate_object_properties(cluster)

            obj = {
                "id": object_id,
                "class": class_name,
                "position": properties["position"].tolist(),
                "dimensions": properties["dimensions"].tolist(),
                "confidence": float(properties["confidence"]),
                "dynamic_probability": DYNAMIC_PROBABILITY.get(class_name, 0.0),
                "num_points": properties["num_points"]
            }

            objects.append(obj)
            object_id += 1

    return objects


objects = detect_objects(
    points,
    labels,
    confidence,
    eps=1.0,
    min_samples=10
)

print("Detected objects:", len(objects))

## 11. Print detected objects

In [ ]:
for obj in objects:
    print("-" * 50)
    print("Object ID:", obj["id"])
    print("Class:", obj["class"])
    print("Position (x,y,z):", np.round(obj["position"], 2))
    print("Dimensions (L,W,H):", np.round(obj["dimensions"], 2))
    print("Confidence:", round(obj["confidence"] * 100, 2), "%")
    print("Dynamic probability:", obj["dynamic_probability"])
    print("Number of points:", obj["num_points"])

## 12. Show results as a table

In [ ]:
rows = []

for obj in objects:
    rows.append({
        "ID": obj["id"],
        "Class": obj["class"],
        "X": obj["position"][0],
        "Y": obj["position"][1],
        "Z": obj["position"][2],
        "Length": obj["dimensions"][0],
        "Width": obj["dimensions"][1],
        "Height": obj["dimensions"][2],
        "Confidence": obj["confidence"],
        "Dynamic Probability": obj["dynamic_probability"],
        "Points": obj["num_points"]
    })

objects_df = pd.DataFrame(rows)
objects_df.round(3)

## 13. Visualize detected object centers

In [ ]:
plt.figure(figsize=(10, 7))

for class_id, class_name in CLASS_NAMES.items():
    mask = labels == class_id

    if np.any(mask):
        plt.scatter(
            points[mask, 0],
            points[mask, 1],
            s=5,
            alpha=0.5,
            label=class_name
        )

for obj in objects:
    x, y, z = obj["position"]

    plt.scatter(
        x,
        y,
        marker="x",
        s=120
    )

    plt.text(
        x + 0.3,
        y + 0.3,
        f'{obj["class"]} #{obj["id"]}'
    )

plt.xlabel("X (meters)")
plt.ylabel("Y (meters)")
plt.title("Detected Object Instances")
plt.legend()
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()

## 14. Create Segment 4 JSON output

In [ ]:
segment4_output = {
    "timestamp": 123.42,
    "objects": objects
}

json_output = json.dumps(
    segment4_output,
    indent=4
)

print(json_output)

## 15. Save structured object data

In [ ]:
output_file = "segment4_objects.json"

with open(output_file, "w") as f:
    json.dump(
        segment4_output,
        f,
        indent=4
    )

print("Saved:", output_file)

## Next step

Once this works with real Segment 3 classified LiDAR points, Segment 5 can consume the structured objects and terrain data to construct the **2.5D map**.

For real data, replace the synthetic-data cell with arrays shaped like:

```python
points.shape      # (N, 3)
labels.shape      # (N,)
confidence.shape  # (N,)
```

where every row of `points` is `[x, y, z]`.
